# End-to-End Pipeline Test

This notebook runs the ingestion pipeline and stores the vectors in the local standalone Qdrant binary. 
**Prerequisite:** Ensure Qdrant is running by opening a new terminal and executing: `./bin/qdrant`

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))


import logging
from configs import config
from main import PipelineOrchestrator

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("e2e_notebook")

/Users/ayushdubey/Source/Trial/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


### 1. Ingestion Phase
This cell initializes the orchestrator, downloads a batch of PDFs from Google Cloud, parses them, chunks them, embeds them, and sends them to the local Qdrant server over port 6333.

In [2]:
# LOWER THE BATCH SIZE to prevent Mac from freezing under memory pressure
config.EMBEDDING_BATCH_SIZE = 32
config.EMBEDDING_DEVICE = 'cpu'

# Initialize Orchestrator (defaults to localhost:6333 for Qdrant)
orchestrator = PipelineOrchestrator(
    download_dir="e2e_trial_batch", 
    state_file="e2e_trial_state.json"
)

# Run a single batch of 5 PDFs
orchestrator.run(max_batches=1, batch_size=5)

Discovering historical arXiv folders...


2026-05-25 13:36:10,490 [ERROR] docling is not installed. Please install it with `pip install docling` to use the LayoutAwareChunker.
/Users/ayushdubey/Source/Trial/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-25 13:36:11,660 [INFO] Loading embedding model 'BAAI/bge-m3' on device 'cpu'...
2026-05-25 13:36:11,959 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-25 13:36:11,961 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-25 13:36:11,996 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 200 OK"
2026-05-

### 2. Verification
Let's directly query the Qdrant database to see how many chunks it currently holds.

In [7]:
info = orchestrator.store._client.get_collection(config.QDRANT_COLLECTION)
print(f"\n✅ Total chunks successfully stored in Qdrant: {info.points_count}")

2026-05-25 12:49:09,909 [INFO] HTTP Request: GET http://localhost:6333/collections/arxiv_chunks "HTTP/1.1 200 OK"



✅ Total chunks successfully stored in Qdrant: 447


### 3. Interactive Search
Change the `query` variable below to anything you want, run the cell, and see the top 3 most relevant chunks retrieved from the database.

In [ ]:
query = "What is the main architecture used in this paper?"
# LOWER THE BATCH SIZE to prevent Mac from freezing under memory pressure

# Convert text query into a 1024-dimension vector using the BGE-M3 model
query_vector = orchestrator.embedder.embed([query])[0]

# Search the vector database for the closest matches
results = orchestrator.store.search(query_vector=query_vector, top_k=3)

print(f"\n🔍 Top 3 search results for: '{query}'\n" + "-"*50)
for i, r in enumerate(results, 1):
    content_preview = r['content'].replace('\n', ' ')[:200] + "..."
    print(f"{i}. Score: {r['score']:.4f}")
    print(f"   Doc ID: {r['metadata'].get('document_id')}")
    print(f"   Preview: {content_preview}\n")

Discovering historical arXiv folders...


2026-05-25 13:25:19,424 [ERROR] docling is not installed. Please install it with `pip install docling` to use the LayoutAwareChunker.
/Users/ayushdubey/Source/Trial/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-25 13:25:20,765 [INFO] Loading embedding model 'BAAI/bge-m3' on device 'cpu'...
2026-05-25 13:25:21,136 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-25 13:25:21,138 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-25 13:25:21,182 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 200 OK"
2026-05-


🔍 Top 3 search results for: 'What is the main architecture used in this paper?'
--------------------------------------------------
1. Score: 0.5199
   Doc ID: e4db36d2585b1139abfcad6741eef67755572342454f0142cb6f8dfafb6b17d4
.../CX/D3/D2/D7 /CX/D2 P (Q, Q T , y, Ω∗), /DB/CW/CX7/D8/CP/D8/CT /D6/CP/CS/CX/CP/B9 /D8/CX/DA /CT 

2. Score: 0.5186
   Doc ID: 105702e562154e5d0ac5242bbb6363149e649f878427aea3aa620af0eb0c51c6
   Preview: and scene analysis. In G. D. Battista and U. Zwick, editors, ESA, volume 2832 of Lecture Notes in Computer Science . Algorithms - ESA 2003, 11th Annual European Symposium, Budapest,Hungary, Springer, ...

3. Score: 0.5167
   Doc ID: f0476169c53543a861e4a90a9dbf3213ccdfe2eb3e93dc1964f35fc2c5a5da0e
   Preview: 2 /D8/CW/CT/CX/D6 /CS/CP/D8/CP /CP/D2/CP/D0/DD/D7/CX/D7/BA /BE/BA /CA /CT/D7/D9/D1/D1/CT /CS QT /CS/CX/D7/D8/D6/CX/CQ/D9/D8/CX/D3/D2/D7 /CP/D2/CS /CP/DA/CT/D6 /CP/CV/CT /D8/D6 /CP/D2/D7/DA/CT/D6/D7/CT...



### 4. Testing a Specific Paper (Attention Is All You Need)
Let's download the famous transformer paper (arXiv:1706.03762) directly, push it through the pipeline, and run some queries against it.

In [4]:
import urllib.request
import os

# 1. Download the paper directly
paper_id = "1706.03762"
pdf_url = f"https://arxiv.org/pdf/{paper_id}.pdf"
download_path = os.path.join("e2e_trial_batch", f"{paper_id}.pdf")

print(f"Downloading paper {paper_id}...")
urllib.request.urlretrieve(pdf_url, download_path)
print(f"Downloaded {paper_id} to e2e_trial_batch/")

# 2. Re-use the Orchestrator's internal components to process it
from arxiv_scholar.ingestion.local import LocalDirectoryReader
reader = LocalDirectoryReader(directory_path="e2e_trial_batch", file_glob=f"{paper_id}.pdf")

for document in reader.read():
    print(f"Chunking {document.metadata.get('filename')}...")
    chunks = list(orchestrator.chunker.chunk(document))
    print(f"Created {len(chunks)} chunks.")
    
    print("Embedding chunks...")
    texts = [c.content for c in chunks]
    vectors = orchestrator.embedder.embed(texts)
    
    print("Storing in Qdrant...")
    orchestrator.store.upsert(chunks, vectors)

print("✅ Paper successfully ingested!")

Downloaded 1706.03762 to e2e_trial_batch/


2026-05-25 13:28:06,663 [WARNING] Docling not available. Falling back to sliding window chunking.


Chunking 1706.03762.pdf...
Created 31 chunks.
Embedding chunks...


Batches: 100%|██████████| 16/16 [00:08<00:00,  1.84it/s]
2026-05-25 13:28:15,397 [INFO] HTTP Request: PUT http://localhost:6333/collections/arxiv_chunks/points?wait=true "HTTP/1.1 200 OK"


Storing in Qdrant...
✅ Paper successfully ingested!


In [5]:
# 3. Search the newly added paper
query = "What are the main components of the Transformer architecture?"

query_vector = orchestrator.embedder.embed([query])[0]
results = orchestrator.store.search(query_vector=query_vector, top_k=3)

print(f"\n🔍 Top 3 search results for: '{query}'\n" + "-"*50)
for i, r in enumerate(results, 1):
    content_preview = r['content'].replace('\n', ' ')[:300] + "..."
    print(f"{i}. Score: {r['score']:.4f}")
    print(f"   Paper: {r['metadata'].get('filename')}")
    print(f"   Preview: {content_preview}\n")

2026-05-25 13:28:21,930 [INFO] HTTP Request: POST http://localhost:6333/collections/arxiv_chunks/points/query "HTTP/1.1 200 OK"



🔍 Top 3 search results for: 'What are the main components of the Transformer architecture?'
--------------------------------------------------
1. Score: 0.5976
   Paper: 1706.03762.pdf
   Preview: sections, we will describe the Transformer, motivate self-attention and discuss its advantages over models such as [17, 18] and [9]. 3 Model Architecture Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 35]. Here, the encoder maps an input sequence of sym...

2. Score: 0.5718
   Paper: 1706.03762.pdf
   Preview: Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Par...

3. Score: 0.5445
   Paper: 1706.03762.pdf
   Preview: conditional computation [32], while also improving model performance in case of the 

In [12]:
print(f"Active Device: {orchestrator.embedder.device}")
print(f"Active Batch Size: {orchestrator.embedder._batch_size}")
print(f"Model Name: {orchestrator.embedder.model_name}")

Active Device: cpu
Active Batch Size: 2
Model Name: BAAI/bge-m3
